# Notebook 1: Data Preparation (Zenodo EmoWOZ)
**Platform: Google Colab (CPU)**

## EmoWOZ Download (do this before running)
1. Go to https://zenodo.org/records/6506504
2. Download **emowoz-dataset.zip** (the main file)
3. Unzip it Ã¢â‚¬â€ you get a folder of `.json` files like `DMAGE2493.json`, `DMAGE2400.json` etc.
4. Upload the entire folder to your Google Drive under: `My Drive/emowoz_llama2/raw/`

You can drag-and-drop the whole folder into Drive from your browser.
There are ~10k JSON files Ã¢â‚¬â€ the upload takes about 5-10 minutes.

## What this notebook does after that:
1. Reads every JSON file from Drive
2. Parses the alternating USER/SYSTEM turn structure
3. Extracts emotion labels and facts from dialog_act
4. Cleans with Gemini Flash
5. Builds DPO preference pairs
6. Saves everything ready for training

In [1]:
!pip install -q tqdm groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/emowoz_llama2'

RAW_DIR  = f'{BASE_DIR}/raw'

os.makedirs(f'{BASE_DIR}/data/train',     exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/val',       exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/test',      exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/dpo_pairs', exist_ok=True)

json_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.json')]
print(f'Found {len(json_files)} dialogue JSON files in {RAW_DIR}')
print('Sample files:', json_files[:5])

Mounted at /content/drive
Found 2 dialogue JSON files in /content/drive/MyDrive/emowoz_llama2/raw
Sample files: ['emowoz-multiwoz.json', 'emowoz-dialmage.json']


In [ ]:
HF_TOKEN     = ''
HF_USERNAME  = 'Winindux'
GROQ_API_KEY = ''

from groq import Groq
scorer = Groq(api_key=GROQ_API_KEY)
print('Groq scorer ready (llama-3.1-8b-instant Ã¢â‚¬â€ 14,400 req/day free).')

Groq scorer ready (llama-3.1-8b-instant Ã¢â‚¬â€ 14,400 req/day free).


In [6]:
import json, os

sample_path = os.path.join(RAW_DIR, json_files[0])
with open(sample_path) as f:
    sample = json.load(f)

dialogue_id = list(sample.keys())[0]
turns = sample[dialogue_id]['log']

print(f'Dialogue: {dialogue_id}')
print(f'Number of turns: {len(turns)}')
print()
for i, t in enumerate(turns):
    has_emotion   = bool(t.get('emotion'))
    has_dialog_act = bool(t.get('dialog_act'))
    print(f'Turn {i}: text="{t["text"][:60]}..."')
    print(f'         has_emotion={has_emotion}  has_dialog_act={has_dialog_act}')
    print()

Dialogue: PMUL2335.json
Number of turns: 10

Turn 0: text="How are you doing? Are there any European restaurants in the..."
         has_emotion=True  has_dialog_act=True

Turn 1: text="Yes, there are 8. What is your price range?..."
         has_emotion=False  has_dialog_act=True

Turn 2: text="Can I get the name and address of one of the most inexpensiv..."
         has_emotion=True  has_dialog_act=True

Turn 3: text="The phone number to the River Bar Steakhouse and Grill is 01..."
         has_emotion=False  has_dialog_act=True

Turn 4: text="Thanks so much. I am also looking for places to go in town. ..."
         has_emotion=True  has_dialog_act=True

Turn 5: text="What type of place are you looking for?..."
         has_emotion=False  has_dialog_act=True

Turn 6: text="I would like to see colleges.  Can you please recommend one ..."
         has_emotion=True  has_dialog_act=True

Turn 7: text="Churchhill College is free and in the west part of town ..."
         has_emotion=False

In [7]:
EMOTION_LABELS = {
    0: 'neutral',
    1: 'fearful',
    2: 'dissatisfied',
    3: 'apologetic',
    4: 'abusive',
    5: 'excited',
    6: 'satisfied'
}


def get_emotion_label(emotion_list):
    """
    The emotion field in USER turns looks like:
    [
        {"annotator": "f89f2efc", "annotation": 0},
        {"annotator": "f92a5e1b", "annotation": 5},
        {"annotator": "55655209", "annotation": 5},
        {"emotion": 5, "sentiment": 2}      <-- FINAL ground truth entry
    ]
    We use the last entry {"emotion": ..., "sentiment": ...} as ground truth.
    If it doesn't exist we fall back to majority vote of annotators.
    """
    if not emotion_list:
        return 'neutral'

    for entry in emotion_list:
        if 'emotion' in entry and 'annotator' not in entry:
            return EMOTION_LABELS.get(entry['emotion'], 'neutral')

    from collections import Counter
    votes = [e['annotation'] for e in emotion_list if 'annotation' in e]
    if votes:
        majority = Counter(votes).most_common(1)[0][0]
        return EMOTION_LABELS.get(majority, 'neutral')

    return 'neutral'


for i, t in enumerate(turns):
    if t.get('emotion'):
        print(f'Turn {i} emotion: {get_emotion_label(t["emotion"])} | raw: {t["emotion"]}')

Turn 0 emotion: neutral | raw: [{'annotator': '68bd033a', 'annotation': 0}, {'annotator': '13de8dba', 'annotation': 0}, {'annotator': 'c522c02d', 'annotation': 0}, {'emotion': 0, 'sentiment': 0}]
Turn 2 emotion: neutral | raw: [{'annotator': '68bd033a', 'annotation': 0}, {'annotator': '13de8dba', 'annotation': 0}, {'annotator': 'c522c02d', 'annotation': 0}, {'emotion': 0, 'sentiment': 0}]
Turn 4 emotion: satisfied | raw: [{'annotator': '68bd033a', 'annotation': 0}, {'annotator': '13de8dba', 'annotation': 6}, {'annotator': 'c522c02d', 'annotation': 6}, {'emotion': 6, 'sentiment': 2}]
Turn 6 emotion: neutral | raw: [{'annotator': '68bd033a', 'annotation': 0}, {'annotator': '13de8dba', 'annotation': 0}, {'annotator': 'c522c02d', 'annotation': 0}, {'emotion': 0, 'sentiment': 0}]
Turn 8 emotion: satisfied | raw: [{'annotator': '68bd033a', 'annotation': 6}, {'annotator': '13de8dba', 'annotation': 6}, {'annotator': 'c522c02d', 'annotation': 6}, {'emotion': 6, 'sentiment': 2}]


In [8]:
def extract_facts_from_dialog_act(dialog_act):
    """
    dialog_act looks like:
    {
        "Restaurant-Inform": [["Addr", "2 Sturton Street"], ["Phone", "01223306306"]],
        "Taxi-Inform":       [["Car", "tesla"]]
    }

    We flatten this into a readable facts string:
    "Restaurant address: 2 Sturton Street. Restaurant phone: 01223306306. Taxi car: tesla."

    We skip slots with value 'none', '?', or empty.
    """
    if not dialog_act:
        return 'No specific facts available.'

    SLOT_NAMES = {
        'Addr':    'address',
        'Phone':   'phone number',
        'Post':    'postcode',
        'Price':   'price range',
        'Name':    'name',
        'Type':    'type',
        'Fee':     'entrance fee',
        'Car':     'car type',
        'Depart':  'departure',
        'Dest':    'destination',
        'Leave':   'leave time',
        'Arrive':  'arrive time',
        'Food':    'cuisine',
        'Area':    'area',
        'Stars':   'star rating',
        'Internet':'internet',
        'Parking': 'parking',
        'Ref':     'booking reference',
        'Ticket':  'ticket price',
        'Time':    'time',
        'Day':     'day',
        'People':  'number of people',
        'Stay':    'nights to stay',
    }

    SKIP_VALUES = {'none', 'None', '?', '', 'free', 'dontcare'}
    SKIP_VALUES = {'none', 'None', '?', ''}

    facts = []
    for act_key, slot_pairs in dialog_act.items():
        if not slot_pairs:
            continue

        domain = act_key.split('-')[0] if '-' in act_key else act_key
        if domain.lower() in ('general',):
            continue

        for slot, value in slot_pairs:
            if value in SKIP_VALUES:
                continue
            slot_name = SLOT_NAMES.get(slot, slot.lower())
            facts.append(f"{domain} {slot_name}: {value}")

    return '. '.join(facts) if facts else 'No specific facts available.'


for i, t in enumerate(turns):
    if t.get('dialog_act'):
        facts = extract_facts_from_dialog_act(t['dialog_act'])
        print(f'Turn {i} facts: {facts}')

Turn 0 facts: Restaurant cuisine: european
Turn 1 facts: Restaurant choice: 8
Turn 2 facts: Restaurant price range: cheap
Turn 3 facts: Restaurant name: River Bar Steakhouse and Grill. Restaurant address: Quayside Off Bridge Street. Restaurant phone number: 01223307030
Turn 4 facts: No specific facts available.
Turn 5 facts: No specific facts available.
Turn 6 facts: Attraction type: college
Turn 7 facts: Attraction area: west part of town. Attraction entrance fee: free. Attraction name: Churchhill College
Turn 8 facts: No specific facts available.
Turn 9 facts: No specific facts available.


In [9]:
from tqdm import tqdm

def process_dialogue_file(filepath):
    """
    Parse one EmoWOZ JSON file into a list of training examples.

    Structure of each file:
    {
      "DMAGE2493": {
        "log": [
          turn_0 (USER),
          turn_1 (SYSTEM),
          turn_2 (USER),
          turn_3 (SYSTEM),
          ...
        ]
      }
    }

    Rules:
    - USER turns have 'emotion' list with annotators
    - SYSTEM turns have empty 'emotion' list and contain 'dialog_act'
    - Skip turns with empty 'text'
    - We create one training example per USERÃ¢â€ â€™SYSTEM pair
    """
    try:
        with open(filepath) as f:
            data = json.load(f)
    except:
        return []

    examples = []

    for dialogue_id, dialogue_data in data.items():
        turns = dialogue_data.get('log', [])
        history = []
        pending_user = None
        accumulated_facts = []

        for turn in turns:
            text = turn.get('text', '').strip()
            emotion_data = turn.get('emotion', [])
            dialog_act   = turn.get('dialog_act', {})

            if not text:
                continue

            is_user = bool(emotion_data)

            if is_user:
                pending_user = {
                    'text':    text,
                    'emotion': get_emotion_label(emotion_data),
                    'history': list(history)
                }
                history.append(f"User: {text}")

            else:
                facts = extract_facts_from_dialog_act(dialog_act)

                if pending_user:
                    examples.append({
                        'dialogue_id': dialogue_id,
                        'inquiry':     pending_user['text'],
                        'emotion':     pending_user['emotion'],
                        'facts':       facts,
                        'history':     pending_user['history'],
                        'response':    text
                    })
                    pending_user = None

                history.append(f"System: {text}")

    return examples


test_file = os.path.join(RAW_DIR, json_files[0])
test_examples = process_dialogue_file(test_file)
print(f'Extracted {len(test_examples)} examples from {json_files[0]}')
print()
for ex in test_examples:
    print(f"  emotion  : {ex['emotion']}")
    print(f"  inquiry  : {ex['inquiry'][:80]}")
    print(f"  facts    : {ex['facts'][:100]}")
    print(f"  response : {ex['response'][:80]}")
    print(f"  history  : {len(ex['history'])} prior turns")
    print()

Streaming output truncated to the last 5000 lines.
  inquiry  : Thank so much for all your help today! Take Care. Goodbye.
  facts    : No specific facts available.
  response : enjoy your time in Cambridge!
  history  : 14 prior turns

  emotion  : neutral
  inquiry  : I'm looking for a boat attraction in the south.
  facts    : Attraction choice: plenty
  response : There are plenty attractions to choose from.
  history  : 0 prior turns

  emotion  : neutral
  inquiry  : Any in the south?
  facts    : Attraction type: boat. Attraction area: the South. Attraction type: other
  response : I am not finding a boat attraction in the South.  There are other types of attra
  history  : 2 prior turns

  emotion  : neutral
  inquiry  : Sure, how about a theatre attraction?
  facts    : Attraction type: theatre. Attraction name: The Junction. Attraction address: Clifton Way. Attraction
  response : The Junction is a theatre in the south side on Clifton Way. Would you like more 
  history  : 4 

In [10]:
print(f'Processing {len(json_files)} dialogue files...')

all_examples = []
failed_files = []

for fname in tqdm(json_files):
    fpath = os.path.join(RAW_DIR, fname)
    try:
        exs = process_dialogue_file(fpath)
        all_examples.extend(exs)
    except Exception as e:
        failed_files.append((fname, str(e)))

print(f'\nTotal examples extracted : {len(all_examples)}')
print(f'Files that failed        : {len(failed_files)}')
if failed_files:
    print('  Failed:', failed_files[:5])

from collections import Counter
emotion_counts = Counter(ex['emotion'] for ex in all_examples)
print('\nEmotion distribution:')
for emotion, count in sorted(emotion_counts.items(), key=lambda x: -x[1]):
    print(f'  {emotion:15s}: {count:5d} ({100*count/len(all_examples):.1f}%)')

Processing 2 dialogue files...


100%|██████████| 2/2 [00:07<00:00,  3.85s/it]


Total examples extracted : 82769
Files that failed        : 0

Emotion distribution:
  neutral        : 58169 (70.3%)
  satisfied      : 17506 (21.2%)
  dissatisfied   :  4801 (5.8%)
  excited        :   963 (1.2%)
  apologetic     :   840 (1.0%)
  fearful        :   391 (0.5%)
  abusive        :    99 (0.1%)


In [11]:

before = len(all_examples)
all_examples = [
    ex for ex in all_examples
    if ex['facts'] != 'No specific facts available.'
    or ex['emotion'] == 'dissatisfied'
]
print(f'After removing fact-empty examples: {len(all_examples)} (removed {before - len(all_examples)})')

before = len(all_examples)
all_examples = [ex for ex in all_examples if len(ex['response'].split()) >= 5]
print(f'After removing very short responses: {len(all_examples)} (removed {before - len(all_examples)})')

After removing fact-empty examples: 54039 (removed 28730)
After removing very short responses: 53475 (removed 564)


In [12]:
import re

NEGATIVE_EMOTIONS = {'dissatisfied', 'fearful', 'abusive', 'apologetic'}
EMPATHY_MARKERS   = {
    'sorry', 'apologize', 'apologies', 'understand', 'unfortunately',
    "i'm sorry", 'i can help', "don't worry", 'of course',
    'concern', 'appreciate', 'i hear you', 'that must be', 'i understand'
}

def has_none_value(text):
    """Detect slot placeholder leakage: 'The address is none.'"""
    return bool(re.search(r'\bis\s+none\b|\bwas\s+none\b|\bare\s+none\b', text, re.IGNORECASE))

def is_fragmented(text):
    """Detect MultiWOZ-style fragmented sentences: 'It is . You are welcome .'"""
    return bool(re.search(r'\w\s+\.\s+\w', text))

def is_cold_for_negative_user(ex):
    """True if a negative-emotion user gets a response with zero empathy markers."""
    if ex['emotion'] not in NEGATIVE_EMOTIONS:
        return False
    resp_lower = ex['response'].lower()
    return not any(marker in resp_lower for marker in EMPATHY_MARKERS)

before = len(all_examples)
all_examples = [ex for ex in all_examples if not has_none_value(ex['response'])]
print(f'After removing none placeholder responses  : {len(all_examples)} (removed {before - len(all_examples)})')

before2 = len(all_examples)
all_examples = [ex for ex in all_examples if not is_fragmented(ex['response'])]
print(f'After removing fragmented responses          : {len(all_examples)} (removed {before2 - len(all_examples)})')

before3 = len(all_examples)
all_examples = [ex for ex in all_examples if not is_cold_for_negative_user(ex)]
print(f'After removing cold responses for neg. users : {len(all_examples)} (removed {before3 - len(all_examples)})')

from collections import Counter
emotion_counts = Counter(ex['emotion'] for ex in all_examples)
neg_total = sum(emotion_counts.get(e, 0) for e in NEGATIVE_EMOTIONS)
total     = len(all_examples)
print(f'Emotion distribution after hard filters ({total} examples):')
for em, cnt in sorted(emotion_counts.items(), key=lambda x: -x[1]):
    print(f'  {em:15s}: {cnt:5d} ({100*cnt/total:.1f}%)')
print(f'Negative-emotion examples : {neg_total} ({100*neg_total/total:.1f}%)')
print(f'Neutral + satisfied       : {total - neg_total} ({100*(total-neg_total)/total:.1f}%)')
ratio = emotion_counts.get('neutral', 0) / max(neg_total, 1)
print(f'Neutral-to-negative ratio : {ratio:.1f}:1')
if ratio > 10:
    print('WARNING: ratio > 10:1 -- strongly recommend oversampling negative-emotion examples.')

After removing none placeholder responses  : 53045 (removed 430)
After removing fragmented responses          : 48856 (removed 4189)
After removing cold responses for neg. users : 45483 (removed 3373)
Emotion distribution after hard filters (45483 examples):
  neutral        : 39960 (87.9%)
  satisfied      :  4191 (9.2%)
  excited        :   702 (1.5%)
  dissatisfied   :   508 (1.1%)
  apologetic     :    66 (0.1%)
  fearful        :    47 (0.1%)
  abusive        :     9 (0.0%)
Negative-emotion examples : 630 (1.4%)
Neutral + satisfied       : 44853 (98.6%)
Neutral-to-negative ratio : 63.4:1


In [14]:
import time, json, os, random
from tqdm import tqdm

CHECKPOINT_PATH = f'{BASE_DIR}/data/scoring_checkpoint.jsonl'
MAX_PER_RUN     = 10000
random.seed(42)

for i, ex in enumerate(all_examples):
    ex['_id'] = i

ex_by_id = {ex['_id']: ex for ex in all_examples}

already_scored_ids = set()
scored = []

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            ex = json.loads(line)
            scored.append(ex)
            already_scored_ids.add(ex['_id'])
    print(f'Loaded {len(scored)} previously scored examples from checkpoint.')
else:
    print('No checkpoint found Ã¢â‚¬â€ starting fresh.')

sampled_indices = random.sample(list(range(len(all_examples))), min(len(all_examples), 10000))
ordered_examples = [ex_by_id[i] for i in sampled_indices]
remaining = [ex for ex in ordered_examples if ex['_id'] not in already_scored_ids]

print(f'Remaining to score: {len(remaining)}')
to_score_now = remaining[:MAX_PER_RUN]

if not to_score_now:
    print('All examples already scored! Skipping scoring loop.')
else:
    errors = 0

    def score_example(ex):
        """
        Ask Groq llama-3.1-8b-instant to score one training example.
        Returns {faithfulness: 1-5, empathy: 1-5} or None on error.
        Retries up to 3 times if rate limited.
        """
        prompt = f"""Score this dialogue system response. Reply ONLY with JSON, nothing else.

User inquiry: {ex['inquiry']}
User emotional state: {ex['emotion']}
Facts provided to system: {ex['facts']}
System response: {ex['response']}

{{"faithfulness": <1-5, where 5=only uses provided facts no hallucination 1=lots of made up info>,
  "empathy": <1-5, where 5=perfectly appropriate warm tone for {ex['emotion']} user 1=completely wrong tone>,
  "fluency": <1-5, where 5=natural complete sentences like a real person, 1=fragmented broken text with spaces before periods like 'It is . You are welcome . Is there'>}}"""

        for attempt in range(3):
            try:
                result = scorer.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=80,
                )

                text = result.choices[0].message.content.strip()
                text = text.replace("```json", "").replace("```", "").strip()

                import re
                match = re.search(r'\{.*?\}', text, re.DOTALL)
                if not match:
                    print("Scorer ERROR: No JSON object found in response:", text[:100])
                    return None
                parsed = json.loads(match.group())
                return parsed
            except Exception as e:
                err = str(e)
                if "rate" in err.lower() or "429" in err:
                    print(f"Rate limited, waiting 10s... (attempt {attempt+1}/3)")
                    time.sleep(10)
                else:
                    print("Scorer ERROR:", err)
                    return None
        return None

    with open(CHECKPOINT_PATH, 'a') as checkpoint_file:
        for i, ex in enumerate(tqdm(to_score_now)):
            scores = score_example(ex)

            if scores:
                ex['faithfulness_score'] = scores.get('faithfulness', 3)
                ex['empathy_score']      = scores.get('empathy', 3)
                ex['fluency_score']      = scores.get('fluency', 3)
            else:
                ex['faithfulness_score'] = 3
                ex['empathy_score']      = 3
                ex['fluency_score']      = 3

            scored.append(ex)
            checkpoint_file.write(json.dumps(ex) + '\n')
            checkpoint_file.flush()

            time.sleep(0.5)

            if (i + 1) % 300 == 0:
                good = sum(1 for e in scored if e['faithfulness_score'] >= 4 and e['empathy_score'] >= 4)
                print(f'  [{i+1}/{len(to_score_now)}] Total scored: {len(scored)} | High-quality: {good} | Errors: {errors}')

    print(f'\nDone. Scored this run: {len(to_score_now)} | Errors: {errors}')

print(f'Total scored across all runs: {len(scored)}')
print(f'Remaining: {len(remaining) - len(to_score_now)}')

Loaded 10000 previously scored examples from checkpoint.
Remaining to score: 0
All examples already scored! Skipping scoring loop.
Total scored across all runs: 10000
Remaining: 0


In [15]:
clean = [
    ex for ex in scored
    if ex.get('faithfulness_score', 0) >= 4
    and ex.get('empathy_score', 0)      >= 4
    and ex.get('fluency_score', 0)      >= 4
]

print(f'Before cleaning : {len(scored)}')
print(f'After cleaning  : {len(clean)} ({100*len(clean)/len(scored):.1f}% kept)')

from collections import Counter
f_dist = Counter(ex['faithfulness_score'] for ex in scored)
e_dist = Counter(ex['empathy_score']      for ex in scored)
print('\nFaithfulness score distribution:', dict(sorted(f_dist.items())))
print('Empathy score distribution:      ', dict(sorted(e_dist.items())))

emotion_dist = Counter(ex['emotion'] for ex in clean)
print('\nEmotion distribution in clean set:')
for em, cnt in sorted(emotion_dist.items(), key=lambda x: -x[1]):
    print(f'  {em:15s}: {cnt}')

Before cleaning : 10000
After cleaning  : 6893 (68.9% kept)

Faithfulness score distribution: {0: 18, 0.75: 1, 0.8: 1, 1: 70, 2: 280, 3: 343, 3.2: 1, 4: 1251, 4.5: 2, 4.8: 1, 5: 8032}
Empathy score distribution:       {0: 11, 1: 127, 2: 717, 2.5: 1, 3: 1890, 4: 2811, 4.5: 3, 4.75: 1, 5: 4439}

Emotion distribution in clean set:
  neutral        : 5925
  satisfied      : 796
  excited        : 114
  dissatisfied   : 33
  apologetic     : 17
  fearful        : 8


In [16]:
import json, os, urllib.request

ESCONV_DISK_PATH = f'{BASE_DIR}/esconv_dataset'
ESCONV_JSON_PATH = f'{BASE_DIR}/esconv/ESConv.json'
ESCONV_TXT_URL   = 'https://huggingface.co/datasets/thu-coai/esconv/resolve/main/train.txt'

esconv_data = []

if os.path.exists(ESCONV_DISK_PATH):
    print(f'Loading ESConv from disk: {ESCONV_DISK_PATH}')
    from datasets import load_from_disk
    ds = load_from_disk(ESCONV_DISK_PATH)
    for split in ['train', 'validation', 'test']:
        if split in ds:
            for ex in ds[split]:
                esconv_data.append(json.loads(ex['text']))
    print(f'Loaded {len(esconv_data)} conversations from Arrow splits')

elif os.path.exists(ESCONV_JSON_PATH):
    print(f'Loading ESConv from JSON: {ESCONV_JSON_PATH}')
    with open(ESCONV_JSON_PATH) as f:
        esconv_data = json.load(f)
    print(f'Loaded {len(esconv_data)} conversations from ESConv.json')

else:
    print('Downloading ESConv train.txt from HuggingFace...')
    tmp = f'{BASE_DIR}/esconv_train.txt'
    os.makedirs(BASE_DIR, exist_ok=True)
    urllib.request.urlretrieve(ESCONV_TXT_URL, tmp)
    with open(tmp) as f:
        esconv_data = [json.loads(line) for line in f if line.strip()]
    print(f'Downloaded {len(esconv_data)} conversations')

print(f'Total ESConv conversations: {len(esconv_data)}')

def esconv_to_examples(conv):
    """Convert one ESConv conversation into training examples.

    Handles both file formats:
      ESConv.json : speaker='seeker'/'supporter', field='content',
                    strategy inside annotation={'strategy': ...}
      train.txt   : speaker='usr'/'sys', field='text',
                    strategy as direct field
    """
    examples    = []
    emotion     = conv.get('emotion_type', 'neutral').strip()
    situation   = conv.get('situation', '').strip()
    if not situation:
        return []

    history      = []
    pending_user = None

    for turn in conv.get('dialog', []):
        speaker = turn.get('speaker', '')
        text = turn.get('content', turn.get('text', '')).strip()
        if not text:
            continue

        if speaker in ('seeker', 'usr'):
            pending_user = {
                'emotion':  emotion,
                'facts':    situation,
                'history':  list(history),
                'inquiry':  text,
            }
            history.append(f'User: {text}')

        elif speaker in ('supporter', 'sys') and pending_user is not None:
            annotation = turn.get('annotation', {}) or {}
            strategy   = annotation.get('strategy', turn.get('strategy', ''))
            examples.append({
                'emotion':  pending_user['emotion'],
                'facts':    pending_user['facts'],
                'history':  pending_user['history'],
                'inquiry':  pending_user['inquiry'],
                'response': text,
                'source':   'esconv',
                'strategy': strategy,
            })
            pending_user = None
            history.append(f'System: {text}')

    return examples

esconv_examples = []
skipped_quality = 0
skipped_filter  = 0

for conv in esconv_data:
    scores    = conv.get('survey_score', {}).get('seeker', {})
    empathy   = int(scores.get('empathy',   0) or 0)
    relevance = int(scores.get('relevance', 0) or 0)
    if empathy < 4 or relevance < 4:
        skipped_quality += 1
        continue
    for ex in esconv_to_examples(conv):
        if has_none_value(ex['response']):
            skipped_filter += 1; continue
        if is_fragmented(ex['response']):
            skipped_filter += 1; continue
        if is_cold_for_negative_user(ex):
            skipped_filter += 1; continue
        esconv_examples.append(ex)

print(f'ESConv examples kept         : {len(esconv_examples)}')
print(f'Skipped (low quality scores) : {skipped_quality} conversations')
print(f'Skipped (hard filters)       : {skipped_filter} examples')

from collections import Counter
esconv_emo = Counter(ex['emotion'] for ex in esconv_examples)
print('\nESConv emotion distribution after filtering:')
for em, cnt in sorted(esconv_emo.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / max(len(esconv_examples), 1)
    print(f'  {em:15s}: {cnt:5d} ({pct:.1f}%)')

before_merge = len(clean)
clean.extend(esconv_examples)
print(f'\nMerged: {before_merge} (EmoWOZ) + {len(esconv_examples)} (ESConv) = {len(clean)} total')

merged_emo = Counter(ex['emotion'] for ex in clean)
neg_total  = sum(merged_emo.get(e, 0) for e in NEGATIVE_EMOTIONS)
total      = len(clean)
print(f'\nEmotion distribution after merge ({total} examples):')
for em, cnt in sorted(merged_emo.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / total
    print(f'  {em:15s}: {cnt:5d} ({pct:.1f}%)')
ratio = merged_emo.get('neutral', 0) / max(neg_total, 1)
print(f'\nNeutral-to-negative ratio after merge: {ratio:.1f}:1')
if ratio > 10:
    print('WARNING: still skewed -- proceed to Layer 4 oversampling below.')
else:
    print('OK: ratio acceptable for training.')

Loading ESConv from disk: /content/drive/MyDrive/emowoz_llama2/esconv_dataset
Loaded 1300 conversations from Arrow splits
Total ESConv conversations: 1300
ESConv examples kept         : 11403
Skipped (low quality scores) : 306 conversations
Skipped (hard filters)       : 50 examples

ESConv emotion distribution after filtering:
  anxiety        :  3001 (26.3%)
  depression     :  2984 (26.2%)
  sadness        :  2729 (23.9%)
  anger          :   997 (8.7%)
  fear           :   794 (7.0%)
  shame          :   385 (3.4%)
  disgust        :   366 (3.2%)
  nervousness    :   123 (1.1%)
  jealousy       :    14 (0.1%)
  guilt          :    10 (0.1%)

Merged: 6893 (EmoWOZ) + 11403 (ESConv) = 18296 total

Emotion distribution after merge (18296 examples):
  neutral        :  5925 (32.4%)
  anxiety        :  3001 (16.4%)
  depression     :  2984 (16.3%)
  sadness        :  2729 (14.9%)
  anger          :   997 (5.4%)
  satisfied      :   796 (4.4%)
  fear           :   794 (4.3%)
  shame      

In [17]:
random.seed(42)
random.shuffle(clean)

n          = len(clean)
train_data = clean[:int(n * 0.8)]
val_data   = clean[int(n * 0.8):int(n * 0.9)]
test_data  = clean[int(n * 0.9):]

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

Train: 14636 | Val: 1830 | Test: 1830


In [18]:

import random
from collections import Counter

POSITIVE_NEUTRAL = {'neutral', 'satisfied', 'excited'}

def is_negative_emotion(ex):
    return ex.get('emotion', 'neutral').lower() not in POSITIVE_NEUTRAL

neg_train     = [ex for ex in train_data if is_negative_emotion(ex)]
pos_train     = [ex for ex in train_data if not is_negative_emotion(ex)]
current_ratio = len(pos_train) / max(len(neg_train), 1)

print('Before oversampling:')
print(f'  Positive/neutral examples : {len(pos_train)}')
print(f'  Negative examples         : {len(neg_train)}')
print(f'  Ratio                     : {current_ratio:.1f}:1')

emo_before = Counter(ex['emotion'] for ex in train_data)
print('\nPer-emotion (before):')
for em, cnt in sorted(emo_before.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / len(train_data)
    print(f'  {em:15s}: {cnt:5d} ({pct:.1f}%)')

TARGET_RATIO = 3.0

if current_ratio > TARGET_RATIO:
    target_neg   = int(len(pos_train) / TARGET_RATIO)
    extra_needed = target_neg - len(neg_train)

    random.seed(42)
    extra      = random.choices(neg_train, k=extra_needed)
    train_data = pos_train + neg_train + extra
    random.shuffle(train_data)

    new_neg_count = len(neg_train) + extra_needed
    new_ratio     = len(pos_train) / max(new_neg_count, 1)

    print(f'\nAfter oversampling (target {TARGET_RATIO}:1):')
    print(f'  Added {extra_needed} extra negative examples')
    print(f'  Positive/neutral examples : {len(pos_train)}')
    print(f'  Negative examples         : {new_neg_count}')
    print(f'  New ratio                 : {new_ratio:.1f}:1')
    print(f'  Total train size          : {len(train_data)}')
else:
    print(f'\nNo oversampling needed -- ratio {current_ratio:.1f}:1 is already <= {TARGET_RATIO}:1')

emo_after = Counter(ex['emotion'] for ex in train_data)
print(f'\nPer-emotion in train after oversampling ({len(train_data)} examples):')
for em, cnt in sorted(emo_after.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / len(train_data)
    print(f'  {em:15s}: {cnt:5d} ({pct:.1f}%)')

Before oversampling:
  Positive/neutral examples : 5509
  Negative examples         : 9127
  Ratio                     : 0.6:1

Per-emotion (before):
  neutral        :  4785 (32.7%)
  anxiety        :  2410 (16.5%)
  depression     :  2378 (16.2%)
  sadness        :  2142 (14.6%)
  anger          :   802 (5.5%)
  fear           :   638 (4.4%)
  satisfied      :   634 (4.3%)
  shame          :   305 (2.1%)
  disgust        :   289 (2.0%)
  nervousness    :    95 (0.6%)
  excited        :    90 (0.6%)
  dissatisfied   :    28 (0.2%)
  jealousy       :    12 (0.1%)
  apologetic     :    11 (0.1%)
  guilt          :     9 (0.1%)
  fearful        :     8 (0.1%)

No oversampling needed -- ratio 0.6:1 is already <= 3.0:1

Per-emotion in train after oversampling (14636 examples):
  neutral        :  4785 (32.7%)
  anxiety        :  2410 (16.5%)
  depression     :  2378 (16.2%)
  sadness        :  2142 (14.6%)
  anger          :   802 (5.5%)
  fear           :   638 (4.4%)
  satisfied      :  

In [19]:

SYSTEM_PROMPT = """You are a compassionate and helpful assistant. Follow these rules strictly:
1. Respond using ONLY the information provided in the Facts section.
2. Never add information that is not present in the Facts.
3. Adapt your tone to match the user's emotional state.
4. Be warm and empathetic while remaining factually accurate."""


def format_llama2(ex, include_response=True):
    """
    Format one example using LLaMA 2 Chat template:
    <s>[INST] <<SYS>>\nsystem\n<</SYS>>\n\nuser message [/INST] assistant response </s>

    Set include_response=False for inference (model generates from here).
    """
    history_str  = '\n'.join(ex.get('history', [])) if ex.get('history') else 'No previous turns.'
    user_content = (
        f"Emotional State: {ex['emotion']}\n"
        f"Facts: {ex['facts']}\n"
        f"Conversation History:\n{history_str}\n"
        f"User: {ex['inquiry']}"
    )
    if include_response:
        return (
            f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
            f"{user_content} [/INST] {ex['response']} </s>"
        )
    return (
        f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"{user_content} [/INST]"
    )


train_fmt = [{'text': format_llama2(ex)}          for ex in train_data]
val_fmt   = [{'text': format_llama2(ex)}          for ex in val_data]
test_fmt  = [{'text': format_llama2(ex), 'raw': ex} for ex in test_data]

print('=== FORMATTED EXAMPLE ===')
print(train_fmt[0]['text'])
print()
print(f'Avg token count (rough): {sum(len(t["text"].split()) for t in train_fmt) // len(train_fmt)}')

=== FORMATTED EXAMPLE ===
<s>[INST] <<SYS>>
You are a compassionate and helpful assistant. Follow these rules strictly:
1. Respond using ONLY the information provided in the Facts section.
2. Never add information that is not present in the Facts.
3. Adapt your tone to match the user's emotional state.
4. Be warm and empathetic while remaining factually accurate.
<</SYS>>

Emotional State: neutral
Facts: Restaurant choice: nine
Conversation History:
No previous turns.
User: I'm looking for an expensive restaurant in the west part of town. [/INST] I have found nine restaurants matching your request. Is there a specific type of food you would like? </s>

Avg token count (rough): 296


In [20]:

REJECTION_PROMPTS = {
    'hallucinated': (
        'Write a response that sounds helpful and uses some facts correctly, '
        'but also invents specific details (names, numbers, times, addresses) '
        'NOT present in the given facts. Make the hallucination subtle.'
    ),
    'emotionally_cold': (
        'Write a response that only uses the given facts and answers the question '
        'correctly, but is completely cold, robotic, and clinical. '
        'Show zero empathy or warmth toward the user.'
    ),
    'irrelevant': (
        'Write a warm, empathetic response that is completely off-topic. '
        'It should NOT answer what the user actually asked. '
        'It should sound caring but miss the point entirely.'
    ),
    'mixed_failure': (
        'Write a warm, empathetic, caring response that invents facts '
        'not present in the given facts section. The tone should be good '
        'but the factual content should be made up.'
    ),
    'verbose': (
        'Write a response that answers correctly using only the given facts, '
        'but is unnecessarily long, repetitive, and rambling. '
        'Repeat information multiple times. Add unnecessary filler phrases. '
        'The response should be at least 3x longer than needed.'
    )
}


def generate_rejected_response(ex, rtype):
    prompt = f"""Generate a deliberately bad dialogue system response for classifier training.

User inquiry: {ex['inquiry']}
User emotional state: {ex['emotion']}
Facts available to the system: {ex['facts']}
Good reference response: {ex['response']}

Task: {REJECTION_PROMPTS[rtype]}

Output ONLY the bad response. No labels, no preamble, no explanation."""
    for attempt in range(3):
            try:
                result = scorer.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=200,
                    timeout=30,
                )
                return result.choices[0].message.content.strip()
            except Exception as e:
                err = str(e)
                if "rate" in err.lower() or "429" in err:
                    print(f"Rate limited, waiting 10s... (attempt {attempt+1}/3)")
                    time.sleep(10)
                else:
                    print("DPO ERROR:", err)
                    return None
    return None








In [21]:
from collections import defaultdict
by_emotion = defaultdict(list)
for ex in train_data:
    by_emotion[ex['emotion']].append(ex)

EMOTION_TARGETS = {
    'neutral':      300,
    'satisfied':    150,
    'excited':       30,
    'dissatisfied':  28,
    'apologetic':    11,
    'fearful':        8,
    'anxiety':      200,
    'depression':   200,
    'sadness':      200,
    'anger':        150,
    'fear':         100,
    'shame':        100,
    'disgust':       50,
}

dpo_pool = []
for emo, target in EMOTION_TARGETS.items():
    available = by_emotion.get(emo, [])
    if not available:
        print(f'WARNING: No examples for {emo}, skipping')
        continue
    if len(available) >= target:
        sampled = random.sample(available, target)
    else:
        print(f'WARNING: Only {len(available)} for {emo}, using all')
        sampled = available
    dpo_pool.extend(sampled)

random.shuffle(dpo_pool)
print(f'Total DPO pool: {len(dpo_pool)}')

DPO_CHECKPOINT = f'{BASE_DIR}/data/dpo_checkpoint.jsonl'

dpo_pairs = []
already_done = 0

if os.path.exists(DPO_CHECKPOINT):
    with open(DPO_CHECKPOINT) as f:
        for line in f:
            dpo_pairs.append(json.loads(line))
    already_done = len(dpo_pairs)
    print(f'Loaded {already_done} previously built DPO pairs from checkpoint.')
else:
    print('No DPO checkpoint found Ã¢â‚¬â€ starting fresh.')

dpo_pool_remaining = dpo_pool[already_done:]
print(f'Remaining to build: {len(dpo_pool_remaining)}')

print(f'Building DPO preference pairs...')

with open(DPO_CHECKPOINT, 'a') as ckpt_file:
    for i, ex in enumerate(tqdm(dpo_pool_remaining)):
        if ex['emotion'] == 'dissatisfied':
            rtype = 'emotionally_cold'
        else:
            rtype = random.choice(list(REJECTION_PROMPTS.keys()))

        rejected = generate_rejected_response(ex, rtype)

        if rejected:
            pair = {
                'prompt':         format_llama2(ex, include_response=False),
                'chosen':         ex['response'].strip(),
                'rejected':       rejected.strip(),
                'rejection_type': rtype,
                'emotion':        ex['emotion'],
            }
            dpo_pairs.append(pair)
            ckpt_file.write(json.dumps(pair) + '\n')
            ckpt_file.flush()

        time.sleep(0.5)

        if (i + 1) % 50 == 0:
            print(f'  [{already_done + i+1}/{len(dpo_pool)}] Pairs built: {len(dpo_pairs)} Ã¢â‚¬â€ checkpoint saved')

type_dist = Counter(p['rejection_type'] for p in dpo_pairs)
emo_dist  = Counter(p['emotion'] for p in dpo_pairs)
print(f'\nTotal DPO pairs: {len(dpo_pairs)}')
print('Type distribution:', dict(type_dist))
print('Emotion distribution:', dict(emo_dist))

Total DPO pool: 1527
No DPO checkpoint found Ã¢â‚¬â€ starting fresh.
Remaining to build: 1527
Building DPO preference pairs...


  3%|▎         | 50/1527 [00:49<24:38,  1.00s/it]

  [50/1527] Pairs built: 50 Ã¢â‚¬â€ checkpoint saved


  7%|▋         | 100/1527 [01:38<24:34,  1.03s/it]

  [100/1527] Pairs built: 100 Ã¢â‚¬â€ checkpoint saved


...

  [1450/1527] Pairs built: 1450 Ã¢â‚¬â€ checkpoint saved


 98%|█████████▊| 1500/1527 [25:04<00:29,  1.08s/it]

  [1500/1527] Pairs built: 1500 Ã¢â‚¬â€ checkpoint saved


100%|██████████| 1527/1527 [25:29<00:00,  1.00s/it]


Total DPO pairs: 1527
Type distribution: {'irrelevant': 266, 'hallucinated': 320, 'verbose': 329, 'emotionally_cold': 310, 'mixed_failure': 302}
Emotion distribution: {'fear': 100, 'neutral': 300, 'depression': 200, 'satisfied': 150, 'excited': 30, 'sadness': 200, 'anger': 150, 'anxiety': 200, 'shame': 100, 'disgust': 50, 'apologetic': 11, 'dissatisfied': 28, 'fearful': 8}


In [22]:

before = len(dpo_pairs)

dpo_pairs = [
    p for p in dpo_pairs
    if not has_none_value(p['chosen'])
    and not is_fragmented(p['chosen'])
    and not (
        p.get('emotion', 'neutral') in NEGATIVE_EMOTIONS
        and not any(m in p['chosen'].lower() for m in EMPATHY_MARKERS)
    )
]

removed = before - len(dpo_pairs)
print(f'Layer 4 DPO chosen filter: removed {removed} bad chosen pairs ({before} Ã¢â€ â€™ {len(dpo_pairs)})')

if removed > 0:
    type_dist = Counter(p['rejection_type'] for p in dpo_pairs)
    emo_dist  = Counter(p['emotion']         for p in dpo_pairs)
    print('Remaining type dist :', dict(type_dist))
    print('Remaining emotion dist:', dict(emo_dist))

Layer 4 DPO chosen filter: removed 0 bad chosen pairs (1527 Ã¢â€ â€™ 1527)


In [23]:
MIN_CLEAN = 300

if len(clean) < MIN_CLEAN:
    print(f'WARNING: Only {len(clean)} clean examples Ã¢â‚¬â€ not saving yet.')
    print('Run more scoring first.')
else:
    def save_jsonl(data, path):
        with open(path, 'w') as f:
            for item in data:
                f.write(json.dumps(item) + '\n')
        size_kb = os.path.getsize(path) / 1024
        print(f'  Saved {len(data):>5} examples Ã¢â€ â€™ {os.path.basename(path)}  ({size_kb:.0f} KB)')

    print('Saving to Google Drive...')
    save_jsonl(train_fmt,  f'{BASE_DIR}/data/train/sft_train.jsonl')
    save_jsonl(val_fmt,    f'{BASE_DIR}/data/val/sft_val.jsonl')
    save_jsonl(test_fmt,   f'{BASE_DIR}/data/test/sft_test.jsonl')
    save_jsonl(dpo_pairs,  f'{BASE_DIR}/data/dpo_pairs/dpo_pairs.jsonl')

    utils_code = '''import json

    SYSTEM_PROMPT = """You are a compassionate and helpful assistant. Follow these rules strictly:
    1. Respond using ONLY the information provided in the Facts section.
    2. Never add information that is not present in the Facts.
    3. Adapt your tone to match the user's emotional state.
    4. Be warm and empathetic while remaining factually accurate."""

    EMOTION_LABELS = {0: "neutral", 1: "fearful", 2: "dissatisfied",
                      3: "apologetic", 4: "abusive", 5: "excited", 6: "satisfied"}

    def format_llama2(ex, include_response=True):
        history_str  = "\\n".join(ex.get("history", [])) if ex.get("history") else "No previous turns."
        user_content = (
            f"Emotional State: {ex[\'emotion\']}\\n"
            f"Facts: {ex[\'facts\']}\\n"
            f"Conversation History:\\n{history_str}\\n"
            f"User: {ex[\'inquiry\']}"
        )
        if include_response:
            return (
                f"<s>[INST] <<SYS>>\\n{SYSTEM_PROMPT}\\n<</SYS>>\\n\\n"
                f"{user_content} [/INST] {ex[\'response\']} </s>"
            )
        return (
            f"<s>[INST] <<SYS>>\\n{SYSTEM_PROMPT}\\n<</SYS>>\\n\\n"
            f"{user_content} [/INST]"  # add space
        )
    '''

    with open(f'{BASE_DIR}/prompt_utils.py', 'w') as f:
        f.write(utils_code)

    summary = {
        'total_dialogues':  len(json_files),
        'total_examples':   len(all_examples),
        'after_cleaning':   len(clean),
        'train':            len(train_data),
        'val':              len(val_data),
        'test':             len(test_data),
        'dpo_pairs':        len(dpo_pairs),
        'emotion_dist':     dict(Counter(ex['emotion'] for ex in clean))
    }
    with open(f'{BASE_DIR}/data_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print('\nSummary:')
    print(json.dumps(summary, indent=2))
    print('\nNotebook 1 complete! Next: Kaggle Notebook 2 (SFT Training)')

Saving to Google Drive...
  Saved 14636 examples Ã¢â€ â€™ sft_train.jsonl  (24095 KB)
  Saved  1830 examples Ã¢â€ â€™ sft_val.jsonl  (2997 KB)
  Saved  1830 examples Ã¢â€ â€™ sft_test.jsonl  (5455 KB)
  Saved  1527 examples Ã¢â€ â€™ dpo_pairs.jsonl  (3446 KB)

Summary:
{
  "total_dialogues": 2,
  "total_examples": 45483,
  "after_cleaning": 18296,
  "train": 14636,
  "val": 1830,
  "test": 1830,
  "dpo_pairs": 1527,
  "emotion_dist": {
    "neutral": 5925,
    "anxiety": 3001,
    "depression": 2984,
    "sadness": 2729,
    "anger": 997,
    "shame": 385,
    "satisfied": 796,
    "fear": 794,
    "disgust": 366,
    "fearful": 8,
    "guilt": 10,
    "excited": 114,
    "nervousness": 123,
    "dissatisfied": 33,
    "jealousy": 14,
    "apologetic": 17
  }
}

Notebook 1 complete! Next: Kaggle Notebook 2 (SFT Training)
